In [1]:
# Load Operational Demand & Withdrawal Data
import sys
sys.path.insert(0, '../src')  # Add src directory to path (go up one level from notebooks/)
from wa_data import load_demand, to_trading_intervals

# Load with explicit path (go up one level from notebooks/)
df = load_demand(raw='../data/raw')

YEAR = 2025   # the year every slice and chart below follows

In [2]:
print(df.head())
print(df.shape)
print(df.columns)

                   ts  operational_demand_mw  unscheduled_demand_mw  \
0 2023-10-01 08:00:00                1606.73                1598.12   
1 2023-10-01 08:05:00                1594.24                1586.53   
2 2023-10-01 08:10:00                1581.83                1574.04   
3 2023-10-01 08:15:00                1554.51                1547.28   
4 2023-10-01 08:20:00                1524.89                1517.86   

   withdrawal_mw  
0          -8.61  
1          -7.71  
2          -7.79  
3          -7.23  
4          -7.03  
(307008, 4)
Index(['ts', 'operational_demand_mw', 'unscheduled_demand_mw',
       'withdrawal_mw'],
      dtype='object')


In [3]:
df_agg = to_trading_intervals(df, ["operational_demand_mw", "unscheduled_demand_mw", "withdrawal_mw"], ts="ts", how="mean")

In [4]:
print(df_agg.head())
print(df_agg.shape)
print(df_agg.columns)

                   ts  operational_demand_mw  unscheduled_demand_mw  \
0 2023-10-01 08:00:00            1560.665000            1553.233333   
1 2023-10-01 08:30:00            1419.940000            1413.580000   
2 2023-10-01 09:00:00            1406.433333            1398.253333   
3 2023-10-01 09:30:00            1257.871667            1252.213333   
4 2023-10-01 10:00:00            1381.716667            1334.556667   

   withdrawal_mw  
0      -7.431667  
1      -6.360000  
2      -8.180000  
3      -5.658333  
4     -47.160000  
(51168, 4)
Index(['ts', 'operational_demand_mw', 'unscheduled_demand_mw',
       'withdrawal_mw'],
      dtype='object')


In [5]:
# Slice the chosen year from the CONCATENATED frame — never from a single-year
# file, which is cut at 00:00 UTC (08:00 AWST) and so misses its own first
# 8 hours of January (they live in the previous year's file).
dyr = df_agg[df_agg.ts.dt.year == YEAR]

In [6]:
print(dyr.head())
print(dyr.shape)
print(dyr.columns)

                       ts  operational_demand_mw  unscheduled_demand_mw  \
39488 2026-01-01 00:00:00            2039.470000            1954.141667   
39489 2026-01-01 00:30:00            2009.188333            1921.446667   
39490 2026-01-01 01:00:00            1937.733333            1897.720000   
39491 2026-01-01 01:30:00            2025.063333            1878.425000   
39492 2026-01-01 02:00:00            2040.713333            1843.473333   

       withdrawal_mw  
39488     -85.328333  
39489     -87.741667  
39490     -40.013333  
39491    -146.638333  
39492    -197.240000  
(11680, 4)
Index(['ts', 'operational_demand_mw', 'unscheduled_demand_mw',
       'withdrawal_mw'],
      dtype='object')


## Visualisation

Three views of the same year, each solving the density problem a different way:

1. **Average day by season** — collapses ~100k intervals into a mean day shape.
2. **One week at 5-minute resolution** — keeps the detail, narrows the window.
3. **Carpet plots** — every interval at once: position for time, colour for magnitude.

Set `YEAR` once in the next cell; everything below follows it.

In [ ]:
# ── Chart setup: palette, shared theme, and derived frames ───────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wa_data import add_time_parts

# Categorical slots 1-3 of the reference palette, in fixed order (never cycled).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"

# Chart chrome, light surface
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")

# Single-hue sequential ramps: blue for demand, orange for charging
RAMP_BLUE   = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
RAMP_ORANGE = ["#fde3d6", "#fbc7ae", "#f8a785", "#f2865c", "#eb6834", "#c9501f", "#73290c"]


def style(fig, title, subtitle=None, height=420, hover="x unified",
          top=108, bottom=58, legend_y=1.0):
    """Shared theme: light surface, recessive grid, muted axes, ink-coloured text."""
    head = f"<b>{title}</b>"
    if subtitle:
        head += f"<br><span style='font-size:12.5px;color:{INK_2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=17, color=INK), x=0, xanchor="left", y=0.96),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, hovermode=hover,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK_2),
        height=height, margin=dict(t=top, r=30, b=bottom, l=74),
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, x=0,
                    bgcolor="rgba(0,0,0,0)", font=dict(color=INK_2)),
    )
    fig.update_xaxes(showgrid=False, linecolor=AXIS, ticks="outside",
                     tickcolor=AXIS, tickfont=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, linecolor="rgba(0,0,0,0)", tickfont=dict(color=MUTED))
    return fig


# The chosen year at both resolutions, with calendar / time-of-day helpers attached.
# d30 enriches the 30-min slice made above; d05 is the matching 5-min slice.
d30 = add_time_parts(dyr.copy())
d05 = add_time_parts(df[df.ts.dt.year == YEAR].copy())
print(f"{YEAR}:  30-min {len(d30):,} rows    5-min {len(d05):,} rows    {d30.date.nunique()} days")
print(f"coverage: {d05.ts.min()}  ->  {d05.ts.max()}")

### 1. The average day, by season

Averaging every day in a season onto one 24-hour axis turns ~100,000 points into 48 per line.
The mirrored layout puts consumer demand above zero and battery charging below it.

Seasons are meteorological and southern-hemisphere, so Summer is Dec–Feb. Any season holding
fewer than `MIN_DAYS` days is dropped — all four survive in a complete year like 2025, but it
matters if you set `YEAR = 2026`, where the data stops on 1 September.

In [ ]:
# ── Chart 1: average day shape by season ─────────────────────────────────────
SEASON_ORDER = ["Summer", "Autumn", "Winter", "Spring"]
MIN_DAYS = 30          # below this, an "average day" is not meaningful

days = d30.groupby("season")["date"].nunique()
seasons = [s for s in SEASON_ORDER if days.get(s, 0) >= MIN_DAYS]
print("days per season:", days.to_dict())
print("plotted:", seasons, " dropped:", [s for s in SEASON_ORDER if s not in seasons])

prof = (d30.groupby(["season", "tod_min"])[["unscheduled_demand_mw", "withdrawal_mw"]]
        .mean().reset_index())

fig = make_subplots(
    rows=1, cols=len(seasons), shared_yaxes=True, horizontal_spacing=0.03,
    subplot_titles=[f"{s} <span style='color:{MUTED};font-weight:400'>· {days[s]} days</span>"
                    for s in seasons])

for i, s in enumerate(seasons, start=1):
    p = prof[prof.season == s].sort_values("tod_min")
    hrs = p.tod_min / 60
    fig.add_trace(go.Scatter(
        x=hrs, y=p.unscheduled_demand_mw, name="Unscheduled demand",
        line=dict(color=BLUE, width=2), legendgroup="d", showlegend=(i == 1),
        hovertemplate="%{y:.0f} MW<extra>Unscheduled demand</extra>"), row=1, col=i)
    fig.add_trace(go.Scatter(
        x=hrs, y=p.withdrawal_mw, name="Storage charging (withdrawal)",
        line=dict(color=ORANGE, width=2), fill="tozeroy",
        fillcolor="rgba(235,104,52,0.16)", legendgroup="w", showlegend=(i == 1),
        hovertemplate="%{y:.0f} MW<extra>Withdrawal</extra>"), row=1, col=i)
    fig.update_xaxes(range=[0, 24], tickvals=[0, 6, 12, 18], row=1, col=i)

fig.update_yaxes(zeroline=True, zerolinecolor=AXIS, zerolinewidth=1.5)
fig.update_yaxes(title_text="MW", row=1, col=1)
for a in fig.layout.annotations[:len(seasons)]:
    a.font.size, a.font.color = 13, INK

style(fig, f"The average day, by season — {YEAR}",
      "Mean of every day in the season, 30-minute intervals. "
      "Consumer demand above the line; grid-scale battery charging below it.",
      height=490, top=145, bottom=88, legend_y=1.13)
# One centred axis label rather than one per panel, so it stays centred at any panel count
fig.add_annotation(text="hour of day", xref="paper", yref="paper", x=0.5, y=-0.22,
                   showarrow=False, font=dict(color=MUTED, size=12))
fig.show()

### 2. One week, at full 5-minute resolution

Averaging hides what makes storage interesting — how sharply it moves. This keeps every 5-minute
reading and narrows the window to a week instead.

`ANCHOR` is the day of 2025's lowest unscheduled demand (386 MW). Change it to look at any other
week: `2025-01-20` is the annual demand peak (4,454 MW), `2025-12-26` the deepest charging day.

In [ ]:
# ── Chart 2: one week in 5-minute detail ─────────────────────────────────────
ANCHOR = pd.Timestamp("2025-11-16")     # lowest unscheduled demand of 2025
start  = ANCHOR - pd.Timedelta(days=3)
wk = d05[(d05.ts >= start) & (d05.ts < start + pd.Timedelta(days=7))]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wk.ts, y=wk.operational_demand_mw, name="Operational demand",
    line=dict(color=BLUE, width=2),
    hovertemplate="%{y:.0f} MW<extra>Operational demand</extra>"))
fig.add_trace(go.Scatter(
    x=wk.ts, y=wk.unscheduled_demand_mw, name="Unscheduled demand",
    line=dict(color=AQUA, width=2),
    hovertemplate="%{y:.0f} MW<extra>Unscheduled demand</extra>"))
fig.add_trace(go.Scatter(
    x=wk.ts, y=wk.withdrawal_mw, name="Storage charging (withdrawal)",
    line=dict(color=ORANGE, width=2), fill="tozeroy", fillcolor="rgba(235,104,52,0.16)",
    hovertemplate="%{y:.0f} MW<extra>Withdrawal</extra>"))

fig.update_yaxes(title_text="MW", zeroline=True, zerolinecolor=AXIS, zerolinewidth=1.5)
fig.update_xaxes(dtick=86400000, tickformat="%a %d %b")
style(fig, f"One week at 5-minute resolution — from {start.date()}",
      "The vertical gap between the two demand lines is exactly the charging drawn below zero.",
      height=460)
fig.show()

### 3. Carpet plots — every interval of the year at once

No aggregation, no window: one column per day, one row per time of day, magnitude as colour.
Seasonal drift reads left-to-right; the daily cycle reads top-to-bottom.

In [ ]:
# ── Chart 3: carpet plots ────────────────────────────────────────────────────
def carpet(frame, col, title, subtitle, ramp, transform=None, unit="MW", height=390):
    v = frame[col] if transform is None else transform(frame[col])
    piv = (frame.assign(_v=v)
           .pivot_table(index="tod_min", columns="date", values="_v", aggfunc="mean")
           .sort_index())
    # Clip the colour range to the 1st-99th percentile: a handful of extreme
    # intervals otherwise compress the ramp and wash out the daily structure.
    lo, hi = np.nanpercentile(piv.values.astype(float), [1, 99])
    fig = go.Figure(go.Heatmap(
        z=piv.values, x=piv.columns, y=piv.index / 60, colorscale=ramp,
        zmin=lo, zmax=hi, zsmooth=False,
        colorbar=dict(title=dict(text=unit, side="top"), outlinewidth=0,
                      thickness=12, tickfont=dict(color=MUTED)),
        hovertemplate="%{x|%d %b} · %{y:.1f}h<br>%{z:.0f} " + unit + "<extra></extra>"))
    style(fig, title, subtitle, height=height, hover="closest")
    fig.update_yaxes(title_text="hour of day", tickvals=[0, 6, 12, 18, 24],
                     range=[0, 24], gridcolor="rgba(0,0,0,0)")
    fig.update_xaxes(showgrid=False, tickformat="%b")
    return fig


carpet(d30, "unscheduled_demand_mw",
       f"Unscheduled demand — every half-hour of {YEAR}",
       "One column per day, one row per time of day. Darker means more demand.",
       RAMP_BLUE).show()

carpet(d30, "withdrawal_mw",
       f"Grid-scale battery charging — every half-hour of {YEAR}",
       "Same layout, absolute charging power. Darker means harder charging.",
       RAMP_ORANGE, transform=lambda s: s.abs()).show()